Rag implementation

In [1]:
from email.quoprimime import decode

import requests
from fontTools.feaLib.ast import simplify_name_attributes

corpus_of_document = [
    "The quick brown fox jumps over the lazy dog.",
    "A journey of a thousand miles begins with a single step.",
    "To be or not to be, that is the question.",
    "All that glitters is not gold.",
    "I think, therefore I am."
]

In [2]:
corpus_of_document

['The quick brown fox jumps over the lazy dog.',
 'A journey of a thousand miles begins with a single step.',
 'To be or not to be, that is the question.',
 'All that glitters is not gold.',
 'I think, therefore I am.']

In [3]:
user_query = "What is the meaning of life?"
document = "The meaning of life is a philosophical question concerning the significance of life or existence in general."

In [4]:
from collections import Counter
import math

In [7]:
user_token=user_query.lower().split(" ")
document_token=document.lower().split(" ")

In [8]:
q_counter = Counter(user_token)
doc_counter = Counter(document_token)

In [15]:
lst=[]
d_list=[]
for token in q_counter.keys():
    lst.append(q_counter[token])
print(lst)

for token in doc_counter.keys():
    d_list.append(doc_counter[token])
print(d_list)

[1, 1, 1, 1, 1, 1]
[2, 1, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [19]:
mylist=[]
for token in doc_counter.keys() and q_counter.keys():
    mylist.append(doc_counter[token] * q_counter[token])

dot_product = sum(mylist)

In [21]:
q_magnitude = math.sqrt(sum(q_counter[token] **2 for token in doc_counter))
d_magnitude = math.sqrt(sum(doc_counter[token] **2 for token in doc_counter))

In [23]:
similarity = dot_product / (q_magnitude * d_magnitude)
print(similarity)

0.6255432421712244


In [49]:
def cosine_similarity(query, document):
    query_token = query.lower().split(" ")
    document_token = document.lower().split(" ")

    q_counter = Counter(query_token)
    doc_counter = Counter(document_token)

    dot_product = sum(q_counter[token] * doc_counter[token] for token in q_counter.keys() & doc_counter.keys())
    q_magnitude = math.sqrt(sum(q_counter[token] ** 2 for token in q_counter))
    d_magnitude = math.sqrt(sum(doc_counter[token] ** 2 for token in doc_counter))

    if q_magnitude == 0 or d_magnitude == 0:
        return 0.0

    similarity = dot_product / (q_magnitude * d_magnitude)
    return similarity

In [50]:
def return_response(query, corpus):
    similarities = []
    for doc in corpus:
        similarity =cosine_similarity(query, doc)
        similarities.append(similarity)
    return corpus_of_document[similarities.index(max(similarities))]

In [64]:
user_input = "What is the meaning of life?"
relevent_document = [
    "The quick brown fox jumps over the lazy dog.",
    "A journey of a thousand miles begins with a single step.",
    "To be or not to be, that is the question.",
    "All that glitters is not gold.",
    "I think, therefore I am."
]

In [65]:
return_response(user_query, relevent_document)

'The quick brown fox jumps over the lazy dog.'

In [66]:
# augument this response by using llama2 model


In [67]:
import json
import requests
full_response = []
prompt= """you are the bot that  makes recommendation for activities. you answer in very shot sentence and you are very friendly. you are given a query and a document. you have to answer the query based on the document. if the document is not relevant to the query, you have to say "I am sorry, I cannot help you with that."""

url = "http://localhost:11434/api/generate"

data = {
    "model" : "llama2",
    "prompt" : prompt.format(user_input=user_input,relevent_document=relevent_document)
}
headers= {
    "Content-Type": "application/json" }
response = requests.post(url, data=json.dumps(data), headers=headers, stream = True)

try:
    for line in response.iter_lines():
        if line:
            decoded_line= json.loads(line.decode('utf-8'))
            if "text" in decoded_line:
                full_response.append(decoded_line["response"])
finally:
    response.close()
print("",join(full_response))


ConnectionError: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError("HTTPConnection(host='localhost', port=11434): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))